# 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

# 2. Load Dataset

In [ ]:
train_df = pd.read_csv("twitter_training.csv")
val_df = pd.read_csv("twitter_validation.csv")

train_df.columns = ['id', 'entity', 'label', 'text']
val_df.columns = ['id', 'entity', 'label', 'text']

train_df = train_df[['text', 'label']]
val_df = val_df[['text', 'label']]

train_df = train_df.sample(2000)
val_df = val_df.sample(min(1000, len(val_df)), random_state=42)

train_df = train_df.sample(2000, random_state=42)
val_df = val_df.sample(500, random_state=42)

In [ ]:
print("Training Data Shape:", train_df.shape)
print("Validation Data Shape:", val_df.shape)

train_df.head()

# 3. Data Preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = " ".join(text.split())
    return text

print("Before Cleaning:", train_df['text'].iloc[0])

train_df.dropna(inplace=True)
val_df.dropna(inplace=True)

train_df['text'] = train_df['text'].apply(clean_text)
val_df['text'] = val_df['text'].apply(clean_text)

print("\nAfter Cleaning:", train_df['text'].iloc[0])

# 4. Missing Values Check

In [ ]:
print("\nMissing Values:\n", train_df.isnull().sum())

# 5. Label Encoding

In [ ]:
labels = train_df['label'].unique().tolist()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

train_df['label'] = train_df['label'].map(label2id)
val_df['label'] = val_df['label'].map(label2id)

In [ ]:
print("\nLabel Mapping:", label2id)
print("\nLabel Distribution:\n", train_df['label'].value_counts())

# 6. Train-Test Split

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    train_df['text'], train_df['label'], test_size=0.2, random_state=42
)

In [ ]:
print("\nTrain Size:", len(train_texts))
print("Test Size:", len(test_texts))

# 7. Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_df['text'])
test_encodings = tokenize(test_texts)

In [ ]:
print("\nTokenized Sample IDs:\n", train_encodings['input_ids'][0])

# 8. Dataset Class

In [ ]:
class TwitterDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TwitterDataset(train_encodings, train_labels)
val_dataset = TwitterDataset(val_encodings, val_df['label'])
test_dataset = TwitterDataset(test_encodings, test_labels)

# 9. DataLoader

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

# 10. Model Setup

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(labels)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

# 11. Training Function

In [ ]:
def train(model, loader):
    model.train()
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# 12. Evaluation Function

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for batch in loader:
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs)

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            true.extend(inputs['labels'].cpu().numpy())

    acc = accuracy_score(true, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(true, preds, average='weighted')
    cm = confusion_matrix(true, preds)

    return acc, precision, recall, f1, cm

# 13. Training Loop

In [ ]:
epochs = 2

for epoch in range(epochs):
    loss = train(model, train_loader)
    acc, precision, recall, f1, _ = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

# 14. Final Test Results

In [ ]:
acc, precision, recall, f1, cm = evaluate(model, test_loader)

print("\nFinal Test Results:")
print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", cm)

# 15. Confusion Matrix Visualization

In [ ]:
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()